# 📦 DataCo Supply Chain Risk Analysis: EDA & Leakage Inspection
**Author:** Person B (@aakritishakya9) & Person A  
**Environment:** Google Colab  
**Objective:** Inspect distributions, check target balance (`Late_delivery_risk`), eliminate feature leakage, and prepare priors for Gaussian Naive Bayes & Logistic Regression.

## 1. Setup & Download Data in Colab (Direct Cloud Download from Hugging Face)
Run the cell below. It downloads the ~95MB DataCo dataset **directly to Google Colab's cloud storage** in ~5 seconds with no logins or tokens needed.

In [ ]:
import os

print("⏳ Downloading DataCo dataset directly to Google Colab cloud storage...")
!mkdir -p ./data_raw
!wget -q --show-progress -O ./data_raw/DataCoSupplyChainDataset.csv "https://huggingface.co/idoyaaran/dataco-supply-chain-model/resolve/main/DataCoSupplyChainDataset.csv"

print("✅ Download complete! The 180,000-row dataset is ready inside Colab.")

## 2. Load and Inspect Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

# Load the dataset with latin-1 encoding
df = pd.read_csv('./data_raw/DataCoSupplyChainDataset.csv', encoding='latin-1')
print(f"✅ Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## 3. Target Variable Analysis: `Late_delivery_risk`
Let's inspect the target class balance. Is there class imbalance?

In [ ]:
target_counts = df['Late_delivery_risk'].value_counts(normalize=True) * 100
print("Target Distribution (%):")
print(target_counts)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Late_delivery_risk')
plt.title("Class Distribution: Late Delivery Risk (0 = On-Time, 1 = Late)")
plt.xlabel("Late Delivery Risk")
plt.ylabel("Count")
plt.show()

# Store priors for Gaussian Naive Bayes
prior_0 = (df['Late_delivery_risk'] == 0).mean()
prior_1 = (df['Late_delivery_risk'] == 1).mean()
print(f"\nCalculated Priors for Gaussian Naive Bayes:")
print(f"P(Y=0) On-Time: {prior_0:.4f}")
print(f"P(Y=1) Late:    {prior_1:.4f}")

## 4. Target Leakage Verification ⚠️
Notice that `Days for shipping (real)` and `Delivery Status` reflect the outcome *after* shipment. Including them would allow the model to cheat and achieve 100% artificial accuracy.

In [ ]:
# Inspect relationship between scheduled shipping days and real shipping days
print("Comparing Delivery Status vs Late_delivery_risk:")
print(pd.crosstab(df['Delivery Status'], df['Late_delivery_risk']))

# Notice that 'Late delivery' status directly maps to risk = 1! We MUST exclude it from features.

## 5. Feature Distributions for Gaussian Naive Bayes
Gaussian Naive Bayes assumes each numerical feature follows a Gaussian (normal) distribution per class. Let's plot the distributions of key order-time features grouped by `Late_delivery_risk`.

In [ ]:
features_to_check = [
    'Days for shipment (scheduled)',
    'Order Item Product Price',
    'Order Item Discount Rate',
    'Order Item Quantity'
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, feat in enumerate(features_to_check):
    sns.kdeplot(data=df, x=feat, hue='Late_delivery_risk', common_norm=False, ax=axes[i], fill=True)
    axes[i].set_title(f'Distribution of {feat} by Late Risk')

plt.tight_layout()
plt.show()

## 6. Shipping Mode Impact
Different shipping modes (Standard Class, First Class, Second Class, Same Day) have vastly different delay probabilities.

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='Shipping Mode', y='Late_delivery_risk', ci=None)
plt.title("Probability of Late Delivery by Shipping Mode")
plt.ylabel("Late Risk Probability (0 to 1)")
plt.show()

## 7. Next Steps: Exporting Clean Data & Starting Model Training
Once EDA is finalized in Colab:
1. Save your notebook directly back to GitHub: **File -> Save a copy in GitHub** -> select your repo `supplier-risk-classification`.
2. Person B (@aakritishakya9) starts training `ScratchGaussianNB` on the preprocessed features.
3. Person A tunes gradient descent iterations and learning rate for `ScratchLogisticRegression`.